1. Install Required Libraries
pip install -r requirements.txt

In [ ]:
import requests
import xmltodict
import pandas as pd

%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../..")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from src.config import ROAD_CLOSURES_URL, ROAD_CLOSURE_HEADERS

closureType = "unplanned"  # Options: "planned", "unplanned"
params = {
    "closureType": closureType,
    "startDateTime": "2024-04-13T00:00:00",
    "endDateTime":   "2024-04-13T23:59:59",
    "modifiedSinceDateTime": "2024-04-13T00:00:00"
}


response = requests.get(ROAD_CLOSURES_URL, headers=ROAD_CLOSURE_HEADERS, params=params)

print("Status Code:", response.status_code)

Status Code: 200


In [12]:
road_closures_dict = xmltodict.parse(response.text)
print(len(road_closures_dict))

1


In [14]:
from src.config import AZURE_CONNECTION_STRING, CONTAINER_ROAD_CLOSURES
from azure.storage.blob import BlobServiceClient

blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_ROAD_CLOSURES)

In [15]:
BATCH_SIZE = 100
UPLOAD_SIZE = 2000

from datetime import datetime

def generate_blob_name(closureType, start_utc):
    if isinstance(start_utc, str):
        start_utc = datetime.fromisoformat(start_utc.replace("Z", "+00:00"))

    timestamp = start_utc.strftime("%Y%m%d_%H%M%S")
    return f"{closureType}_{timestamp}.csv"



In [16]:
from io import StringIO

def upload_to_azure(records, blob_name):
    df = pd.DataFrame(records)

    buffer = StringIO()
    df.to_csv(buffer, index=False)

    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(buffer.getvalue(), overwrite=True)

    print(f"Uploaded {len(records)} records → {blob_name}")

In [17]:
import pandas as pd

def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    return []


# --------------------------------------------------------------
# 1. Extract Road Name
# --------------------------------------------------------------
def get_road_name_from_location(loc_ref):

    # Case A: locSingleRoadLinearLocation (most precise)
    single = loc_ref.get("locSingleRoadLinearLocation")
    if single:
        for lw in ensure_list(single.get("linearWithinLinearElement")):
            try:
                name = (
                    lw.get("linearElement", {})
                      .get("locLinearElementByCode", {})
                      .get("roadName")
                )
                if name:
                    return name
            except:
                pass

    # Case B: grouped locations
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        name = get_road_name_from_location(g)
        if name:
            return name

    # Case C: point locations on linear reference
    point = loc_ref.get("locPointLocation")
    if point:
        for pl in ensure_list(point.get("pointAlongLinearElement")):
            try:
                name = (
                    pl.get("linearElement", {})
                      .get("locLinearElementByCode", {})
                      .get("roadName")
                )
                if name:
                    return name
            except:
                pass

    return None


# --------------------------------------------------------------
# 2. Extract number of lanes closed
# --------------------------------------------------------------
def get_lanes_restricted(loc_ref):

    # Case A: locLinearLocation -> supplementaryPositionalDescription
    lin = loc_ref.get("locLinearLocation")
    if lin:
        sup = lin.get("supplementaryPositionalDescription", {})
        car = ensure_list(sup.get("carriageway", []))

        for c in car:
            impact = (
                c.get("carriagewayExtensionG", {})
                  .get("impactOnCarriageway", {})
                  .get("numberOfLanesRestricted")
            )
            if impact is not None:
                return impact

    # Case B: grouped locations
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        impact = get_lanes_restricted(g)
        if impact:
            return impact

    # Case C: fallback using lane-level closure counts
    if lin:
        sup = lin.get("supplementaryPositionalDescription", {})
        car = ensure_list(sup.get("carriageway", []))

        for c in car:
            lanes = ensure_list(c.get("lane", []))
            closed_count = 0

            for ln in lanes:
                status = (
                    ln.get("laneExtensionG", {})
                      .get("impactOnLanes", {})
                      .get("impactExtensionG", {})
                      .get("lanesStatus")
                )
                if str(status).lower() == "closed":
                    closed_count += 1

            if closed_count > 0:
                return closed_count

    return None

In [18]:
# --------------------------------------------------------------
# Extract coordinates (lat, lon) from posList
# --------------------------------------------------------------
def get_closure_coordinates(loc_ref):
    coords = []

    # Case A: locLinearLocation
    lin = loc_ref.get("locLinearLocation")
    if lin:
        gml = (
            lin.get("gmlLineString", {})
               .get("locGmlLineString", {})
               .get("posList")
        )
        if gml:
            nums = list(map(float, gml.split()))
            pairs = [(nums[i], nums[i+1]) for i in range(0, len(nums), 2)]
            coords.extend(pairs)

    # Case B: multiple locations
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        coords.extend(get_closure_coordinates(g))

    return coords

In [19]:
def get_poslist(loc_ref):
    poslists = []

    # A. Single linear location
    lin = loc_ref.get("locLinearLocation")
    if lin:
        gml = (lin.get("gmlLineString", {})
                  .get("locGmlLineString", {})
                  .get("posList"))
        if gml:
            poslists.append(gml)

    # B. Multiple segments
    group_list = (
        loc_ref.get("locLocationGroupByList", {})
              .get("locationContainedInGroup", [])
    )
    for g in ensure_list(group_list):
        gml = get_poslist(g)
        if gml:
            if isinstance(gml, list):
                poslists.extend(gml)
            else:
                poslists.append(gml)

    return poslists if poslists else None

In [20]:
from datetime import datetime
import pandas as pd

# =========================
# MAIN EXTRACTION + SAVE
# =========================
def extract_road_closures(data_dict):
    all_records = []
    chunk_counter = 0
    current_blob = generate_blob_name(closureType, params['startDateTime'])
    situations = ensure_list(data_dict.get('D2Payload', {}).get('situation', []))
    print(situations,"situations")
    for s in situations:
        situation_id = s.get('idG')
        situation_records = ensure_list(s.get('situationRecord'))

        for rec in situation_records:

            if not isinstance(rec, dict):
                continue

            sm = rec.get('sitRoadOrCarriagewayOrLaneManagement', {})

            validity = sm.get('validity', {})
            time_spec = validity.get('validityTimeSpecification', {})
            cause = sm.get('cause', {})
            loc_ref = sm.get('locationReference', {})

            # Extract fields
            road_name = get_road_name_from_location(loc_ref)
            lanes_closed = get_lanes_restricted(loc_ref)
            poslist = get_poslist(loc_ref)
            coords = get_closure_coordinates(loc_ref)

            # Compute centroid
            if coords:
                lats = [c[0] for c in coords]
                lons = [c[1] for c in coords]
                closure_lat = sum(lats) / len(lats)
                closure_lon = sum(lons) / len(lons)
            else:
                closure_lat = None
                closure_lon = None

            # Create record
            record = {
                "situation_id": situation_id,
                "record_id": sm.get("idG"),
                "start_time": time_spec.get("overallStartTime"),
                "end_time": time_spec.get("overallEndTime"),
                "validity_status": validity.get("validityStatus"),
                "cause_type": cause.get("causeType"),
                "source": sm.get("source", {}).get("sourceIdentification"),
                "road_name": road_name,
                "lanes_closed": lanes_closed,
                "closure_lat": closure_lat,
                "closure_lon": closure_lon,
                "poslist": poslist,

                # Added (important for consistency)
                "ingestion_time": datetime.utcnow()
            }

            # buffer.append(record)
            all_records.append(record)

            # =========================
            # BATCH SAVE (Darwin style)
            # =========================

            if len(all_records) >= BATCH_SIZE:
                upload_to_azure(all_records, current_blob)

                chunk_counter += len(all_records)
                all_records.clear()

                # File rotation
                if chunk_counter >= UPLOAD_SIZE:
                    print("Chunk limit reached → rotating file")
                    current_blob = generate_blob_name(closureType, params['startDateTime'])
                    chunk_counter = 0

    # Save remaining records
    if all_records:
        upload_to_azure(all_records, current_blob)
        print(f"Saved {len(all_records)} records")

    print("Road closures extraction + save completed")
    return pd.DataFrame(all_records)

In [21]:
road_closures_df = extract_road_closures(road_closures_dict)


[] situations
Road closures extraction + save completed


In [22]:
print("Shape:", road_closures_df.shape)
road_closures_df.head()

Shape: (0, 0)


""
